In [8]:
import pandas as pd
import torch
import sys
sys.path.insert(0, '../neuralsd')
from iterative_nsd import IterativeNSD

In [9]:
df = pd.read_csv('arcene_preproc.csv')
df.head()

,0,1,2,3,4,5,6,7,8,9,...,9991,9992,9993,9994,9995,9996,9997,9998,9999,Target
0,0.0,3.0,0.0,0.0,0.0,3.0,1.0,1.0,0.0,0.0,...,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1
1,0.0,2.0,1.0,0.0,1.0,3.0,1.0,0.0,1.0,0.0,...,3.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0,2.0,-1
2,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,0.0,0.0,...,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1
3,0.0,2.0,1.0,1.0,0.0,3.0,2.0,0.0,0.0,0.0,...,3.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,3.0,1
4,2.0,0.0,2.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,-1


In [10]:
df = df.astype(str)

In [11]:
torch.set_default_device("cuda:0")

In [12]:
nsd = IterativeNSD(k=1, quality_measure="WRAcc")
nsd.fit(df, target_as_tuple=("Target", "1"), max_iterations = 10, min_quality=0.07, weight_init="best", additional_params_for_weight_init={"n":1}, verbose=False)

[[0.05370263678645769,
  0.07694726046353993,
  0.09227248120124867,
  0.10132837581994912,
  0.10646699934554338,
  0.12360673556463281,
  0.12959147581873773,
  0.13020786280171773,
  0.13198727996218884,
  0.1329350014604153,
  0.13439522653766015,
  0.13582518982630515,
  0.13609642866222332,
  0.13623978171932102,
  0.1363154561664947,
  0.13635539325383714,
  0.136376466394646,
  0.13638758475354915,
  0.13639345058021643,
  0.13639654518501187,
  0.13639817776470384,
  0.13639903903600514,
  0.13639949339974958,
  0.1363997330988093,
  0.13639985955163356,
  0.13639992626155148,
  0.13639996145421096,
  0.13639998002000409,
  0.13639998981437873,
  0.13639999498137065,
  0.13639999770723396,
  0.13639999914524986,
  0.1363999999038881,
  0.13640000030412058,
  0.13640000051524462,
  0.13640000062664825,
  0.13640000062664825],
 [0.0324545012197502,
  0.04639896941311174,
  0.05563489185895522,
  0.06114023926776868,
  0.06467246116750432,
  0.08079906093745189,
  0.0844072494158

In [13]:
nsd.get_subgroup_descriptions()

[([('1761', '0.0'),
   ('5642', '0.0'),
   ('6160', '0.0'),
   ('6522', '0.0'),
   ('7211', '0.0'),
   ('7259', '0.0'),
   ('8266', '0.0'),
   ('8848', '0.0')],
  0.13640000071640523,
  54.999999986040415,
  7.999999997966574),
 ([('203', '0.0'),
   ('309', '0.0'),
   ('2294', '0.0'),
   ('2594', '0.0'),
   ('3701', '0.0'),
   ('6254', '0.0'),
   ('7193', '1.0'),
   ('8082', '0.0'),
   ('9751', '0.0')],
  0.08689860911377686,
  15.999999997802734,
  0.9999999998573259),
 ([('636', '0.0'),
   ('959', '0.0'),
   ('1994', '0.0'),
   ('2615', '0.0'),
   ('4320', '0.0'),
   ('5048', '0.0'),
   ('5303', '0.0'),
   ('5791', '0.0'),
   ('6793', '0.0'),
   ('8006', '1.0'),
   ('8440', '0.0')],
  0.07868055424701804,
  10.999999995294083,
  2.3186523211948307e-15)]

In [14]:
metrics = [ (tp,fp) for _,_,tp,fp in nsd.get_subgroup_descriptions()]
tps,fps = zip(*metrics)
tps = pd.Series(tps)
tps = tps.apply(lambda x: round(x)).astype(int)
fps = pd.Series(fps)
fps = fps.apply(lambda x: round(x)).astype(int)
tp = tps.sum()
fp = fps.sum()
TP = df[(df['Target']=='1')].shape[0]
FP = df[(df['Target']=='-1')].shape[0]
print([(t,f) for t,f in zip(tps,fps)])
print("Total True Positives:", tp, "out of", TP)
print("Total False Positives:", fp, "out of", FP)
print("Youden's J statistic:", (tp/TP) - (fp/FP))


[(55, 8), (16, 1), (11, 0)]
Total True Positives: 82 out of 88
Total False Positives: 9 out of 112
Youden's J statistic: 0.8514610389610389
